# kna - Korean National Assembly Database - Tutorial

This notebook demonstrates the kna - Korean National Assembly Database, covering the full legislative pipeline of the Korean National Assembly (국회) across assemblies 17-22 (2004-present).

**Data sources**: 열린국회정보 Open API (8 endpoints combined per assembly)

**Coverage**:
- 110,778 bills across 6 assemblies
- 572,127 committee meeting records
- 15,325 judiciary committee meeting records
- 2.4M member-level roll call votes
- 936 legislator-term DW-NOMINATE ideal points (20-22대)

**Contents**:
1. Data Loading & Schema Overview
2. Cross-Assembly Overview (17-22대)
3. Legislative Funnel (22대)
4. Proposer Type Analysis
5. Committee Bottleneck
6. Legislative Inflation
7. Temporal Patterns
8. DW-NOMINATE Ideal Points
9. Roll Call Votes
10. Combining Ideal Points with Bill Lifecycle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import warnings

try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    plt.style.use('seaborn-v0_8-whitegrid')

# Korean font support (auto-detect)
_ko_font = None
for candidate in ['Apple SD Gothic Neo', 'Nanum Gothic', 'NanumGothic', 'Malgun Gothic', 'NanumBarunGothic']:
    if any(candidate in f.name for f in fm.fontManager.ttflist):
        _ko_font = candidate
        break

if _ko_font:
    plt.rcParams['font.family'] = _ko_font
    print(f'Korean font: {_ko_font}')
else:
    print('WARNING: No Korean font found. Korean text may not render correctly.')
    print('Install: pip install fonts-nanum  OR  apt install fonts-nanum-gothic')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (7, 4.5)

# Okabe-Ito colorblind-safe palette
OI = ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7', '#000000']

warnings.filterwarnings('ignore', category=FutureWarning)

DATA_DIR = 'data/processed'


---
## 1. Data Loading & Schema Overview

We load the full master tables for all 6 assemblies (17-22대) and combine them using the 47 columns shared across all assemblies. Each assembly's master table merges 8 API endpoints into a single row-per-bill lifecycle record.

In [ ]:
# Common columns shared across all 6 assemblies
COMMON_COLS = [
    'bill_id', 'bill_no', 'age', 'bill_kind', 'bill_nm',
    'ppsr_kind', 'proposer_text', 'rst_proposer', 'rst_mona_cd',
    'publ_proposer', 'publ_mona_cd',
    'ppsl_dt', 'committee_dt', 'bdg_cmmt_dt',
    'cmt_present_dt', 'cmt_proc_dt', 'cmt_proc_result_cd',
    'jrcmit_prsnt_dt', 'jrcmit_cmmt_dt', 'jrcmit_proc_dt', 'jrcmit_proc_rslt',
    'law_submit_dt', 'law_cmmt_dt', 'law_present_dt', 'law_prsnt_dt',
    'law_proc_dt', 'law_proc_result_cd', 'law_proc_rslt',
    'rgs_prsnt_dt', 'rgs_rsln_dt', 'rgs_conf_nm', 'rgs_conf_rslt',
    'gvrn_trsf_dt', 'prom_dt', 'prom_no', 'prom_law_nm',
    'proc_dt', 'proc_rslt', 'status', 'passed', 'enacted',
    'days_to_proc', 'committee_nm', 'committee_id',
    'jrcmit_nm', 'link_url', 'member_list',
]

# Load and combine all assemblies
frames = []
for age in range(17, 23):
    df = pd.read_parquet(f'{DATA_DIR}/master_bills_{age}.parquet')
    # Use only common columns (some assemblies have extra columns)
    cols_available = [c for c in COMMON_COLS if c in df.columns]
    frames.append(df[cols_available])

all_bills = pd.concat(frames, ignore_index=True)
print(f'Total bills loaded: {len(all_bills):,}')
print(f'Assemblies: {sorted(all_bills["age"].unique())}')
print(f'Columns: {all_bills.shape[1]}')
print(f'Date range: {all_bills["ppsl_dt"].min().strftime("%Y-%m-%d")} ~ {all_bills["ppsl_dt"].max().strftime("%Y-%m-%d")}')

In [ ]:
# Bills per assembly
print('Bills per assembly:')
print(all_bills.groupby('age').size().to_string())

In [ ]:
# Key lifecycle columns and their coverage across all assemblies
lifecycle_cols = {
    'ppsl_dt': 'Proposal (발의)',
    'committee_dt': 'Committee referral (소관위 회부)',
    'cmt_present_dt': 'Committee tabling (소관위 상정)',
    'cmt_proc_dt': 'Committee decision (소관위 처리)',
    'law_submit_dt': 'Judiciary referral (법사위 회부)',
    'law_proc_dt': 'Judiciary decision (법사위 처리)',
    'rgs_prsnt_dt': 'Plenary tabling (본회의 상정)',
    'rgs_rsln_dt': 'Plenary vote (본회의 의결)',
    'proc_dt': 'Final decision (최종 처리)',
    'prom_dt': 'Promulgation (공포)',
}

n = len(all_bills)
coverage = pd.DataFrame([
    {'Stage': label, 'Non-null': all_bills[col].notna().sum(),
     'Coverage (%)': f"{all_bills[col].notna().sum() / n * 100:.1f}%"}
    for col, label in lifecycle_cols.items()
])
coverage

---
## 2. Cross-Assembly Overview (17-22대)

The Korean National Assembly has experienced dramatic legislative inflation over the past two decades. The number of bills introduced per term has more than tripled from the 17th (8,369) to the 21st assembly (26,711), while passage rates have fallen sharply. The 22nd assembly is still in session, so its numbers are provisional.

In [ ]:
# Summary statistics per assembly
summary = all_bills.groupby('age').agg(
    n_bills=('bill_id', 'size'),
    n_laws=('bill_kind', lambda x: (x == '법률안').sum()),
    passed_rate=('passed', 'mean'),
    enacted_rate=('enacted', 'mean'),
    median_days=('days_to_proc', 'median'),
).reset_index()

summary['passed_rate'] = (summary['passed_rate'] * 100).round(1)
summary['enacted_rate'] = (summary['enacted_rate'] * 100).round(1)
summary['median_days'] = summary['median_days'].round(0)

summary.columns = ['Assembly', 'Total Bills', 'Laws (법률안)', 
                    'Passed Rate (%)', 'Enacted Rate (%)', 'Median Days to Decision']
summary

In [ ]:
# Stacked bar chart: bills per assembly by bill_kind
# Group minor types into 'Other'
major_kinds = ['법률안', '결의안', '동의안', '예산안']
all_bills['bill_kind_grouped'] = all_bills['bill_kind'].where(
    all_bills['bill_kind'].isin(major_kinds), 'Other'
)

kind_counts = (all_bills.groupby(['age', 'bill_kind_grouped']).size()
               .unstack(fill_value=0)
               .reindex(columns=['법률안', '결의안', '동의안', '예산안', 'Other']))

fig, ax = plt.subplots(figsize=(7, 4))
kind_colors = {'법률안': OI[3], '결의안': OI[0], '동의안': OI[2], '예산안': OI[4], 'Other': OI[5]}
labels = [f'{a}대' for a in kind_counts.index]
bottom = np.zeros(len(kind_counts))

for kind in ['법률안', '결의안', '동의안', '예산안', 'Other']:
    vals = kind_counts[kind].values
    ax.bar(labels, vals, bottom=bottom, label=kind, 
           color=kind_colors[kind], edgecolor='white', linewidth=0.5)
    bottom += vals

# Annotate totals
for i, total in enumerate(bottom):
    ax.text(i, total + 300, f'{int(total):,}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Number of bills')
ax.set_ylim(0, bottom.max() * 1.12)
ax.legend(fontsize=7, ncol=3, loc='upper left')

# Mark 22대 as in-progress
ax.annotate('in session', xy=(5, kind_counts.loc[22].sum()),
            xytext=(5, kind_counts.loc[22].sum() + 2200),
            ha='center', fontsize=7, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Line chart: enacted rate trend
rates = all_bills.groupby('age').agg(
    passed_rate=('passed', 'mean'),
    enacted_rate=('enacted', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(rates['age'], rates['passed_rate'] * 100, 'o-', color=OI[3],
        label='Broad (passed)', markersize=6, linewidth=1.5)
ax.plot(rates['age'], rates['enacted_rate'] * 100, 's--', color=OI[4],
        label='Narrow (enacted)', markersize=6, linewidth=1.5)

# Annotate enacted rates
for _, row in rates.iterrows():
    ax.annotate(f"{row['enacted_rate']*100:.1f}%",
                (row['age'], row['enacted_rate']*100),
                textcoords='offset points', xytext=(0, -14),
                ha='center', fontsize=7, color=OI[4])

# Mark 22대 as in-progress
ax.axvspan(21.7, 22.3, alpha=0.1, color='gray')

ax.set_xticks(rates['age'])
ax.set_xticklabels([f"{a}대" for a in rates['age']])
ax.set_ylabel('Rate (%)')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(0, 60)

plt.tight_layout()
plt.show()

The enacted rate has fallen from 30.4% in the 17th assembly to 8.1% in the 22nd (still in session). Even the broader "passed" definition (which counts content absorbed into committee alternatives) shows a decline from 53.3% to 26.2%. This reflects a structural shift: more bills are introduced but fewer clear the legislative pipeline.

---
## 3. Legislative Funnel (22대 as example)

The legislative funnel shows stage-by-stage survival from proposal to promulgation. Most bills die in committee and never reach the plenary floor. We use the 22nd assembly as the running example.

In [ ]:
# Load 22nd assembly master for the funnel
bills22 = pd.read_parquet(f'{DATA_DIR}/master_bills_22.parquet')
N = len(bills22)

funnel_stages = [
    ('발의 (Proposal)', N),
    ('소관위 회부 (Committee referral)',
     bills22[['committee_dt', 'jrcmit_cmmt_dt']].notna().any(axis=1).sum()),
    ('소관위 상정 (Committee tabling)',
     bills22[['cmt_present_dt', 'jrcmit_prsnt_dt']].notna().any(axis=1).sum()),
    ('소관위 처리 (Committee decision)',
     bills22[['cmt_proc_dt', 'jrcmit_proc_dt']].notna().any(axis=1).sum()),
    ('법사위 회부 (Judiciary referral)',
     bills22[['law_submit_dt', 'law_cmmt_dt']].notna().any(axis=1).sum()),
    ('본회의 상정 (Plenary tabling)',
     bills22['rgs_prsnt_dt'].notna().sum()),
    ('가결 (Enacted)',
     int(bills22['enacted'].sum())),
    ('공포 (Promulgated)',
     bills22['prom_dt'].notna().sum()),
]

labels, values = zip(*funnel_stages)

fig, ax = plt.subplots(figsize=(8, 5))

y_positions = np.arange(len(labels) - 1, -1, -1)
bar_colors = [OI[i % len(OI)] for i in range(len(labels))]

bars = ax.barh(y_positions, values, height=0.6,
               color=bar_colors, alpha=0.85, edgecolor='white', linewidth=0.5)

# Annotate counts and percentages
for y, val in zip(y_positions, values):
    ax.text(val + N * 0.01, y,
            f'{val:,}  ({val/N*100:.1f}%)',
            va='center', ha='left', fontsize=8, fontweight='bold')

ax.set_yticks(y_positions)
ax.set_yticklabels(labels[::-1], fontsize=8.5)
ax.set_xlim(0, N * 1.25)
ax.set_xlabel('Number of bills')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

In [ ]:
# Funnel summary table
funnel_df = pd.DataFrame(funnel_stages, columns=['Stage', 'Count'])
funnel_df['% of Total'] = (funnel_df['Count'] / N * 100).round(1)
funnel_df['Drop-off'] = funnel_df['Count'].diff().fillna(0).astype(int)
funnel_df['Survival from Previous (%)'] = (
    funnel_df['Count'] / funnel_df['Count'].shift(1) * 100
).round(1)
funnel_df

---
## 4. Proposer Type Analysis

Korean bills are introduced by four main proposer types: legislators (의원), the government (정부), committee chairs (위원장), and the speaker (의장). Their passage rates differ dramatically. Government bills enjoy a structural advantage in the legislative process, while committee-chair bills are omnibus alternatives that consolidate multiple legislator proposals, passing at near-100% rates by design.

In [ ]:
# Enacted rates by proposer type across assemblies
proposer_types = ['의원', '정부', '위원장']
prop_data = []
for age in range(17, 23):
    sub = all_bills[all_bills['age'] == age]
    for ppsr in proposer_types:
        subset = sub[sub['ppsr_kind'] == ppsr]
        if len(subset) > 0:
            prop_data.append({
                'age': age, 'ppsr_kind': ppsr,
                'enacted_rate': subset['enacted'].mean() * 100,
                'n': len(subset)
            })

prop_df = pd.DataFrame(prop_data)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(6)  # 6 assemblies
w = 0.25
colors = {'의원': OI[3], '정부': OI[0], '위원장': OI[2]}

for i, ppsr in enumerate(proposer_types):
    sub = prop_df[prop_df['ppsr_kind'] == ppsr]
    ax.bar(x + (i - 1) * w, sub['enacted_rate'].values, w,
           label=ppsr, color=colors[ppsr], edgecolor='white', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([f'{a}대' for a in range(17, 23)])
ax.set_ylabel('Enacted rate (%)')
ax.legend(fontsize=8)
ax.set_ylim(0, 110)

# Add a horizontal reference line for overall average
overall = all_bills['enacted'].mean() * 100
ax.axhline(overall, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)
ax.text(5.5, overall + 1.5, f'Overall avg: {overall:.1f}%', fontsize=7, color='gray', ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Summary table: proposer type enacted rates
prop_pivot = prop_df.pivot(index='ppsr_kind', columns='age', values='enacted_rate').round(1)
prop_pivot.columns = [f'{a}대' for a in prop_pivot.columns]
prop_pivot.index.name = 'Proposer Type'
prop_pivot

Legislator-sponsored bills have seen their enacted rate decline from 13.2% (17대) to 3.3% (22대), driven by the explosion in the number of proposals. Government bills maintain high passage rates (38-62%), reflecting their role as administration priorities with pre-negotiated committee support. Committee-chair bills pass at near-100% by construction.

---
## 5. Committee Bottleneck

Standing committees are the primary bottleneck in the Korean legislative process. Passage rates vary substantially by committee, reflecting both the volume of bills referred and each committee's political dynamics. We combine assemblies 20-22 for adequate sample sizes.

In [ ]:
# Committee passage rates (20-22대 combined)
recent = all_bills[all_bills['age'].isin([20, 21, 22])].copy()

cmt_stats = (recent[recent['committee_nm'].notna()]
             .groupby('committee_nm')
             .agg(n_bills=('bill_id', 'size'),
                  enacted_rate=('enacted', 'mean'))
             .reset_index())

# Filter to committees with at least 500 bills for meaningful comparison
cmt_stats = cmt_stats[cmt_stats['n_bills'] >= 500].sort_values('enacted_rate', ascending=True)

fig, ax = plt.subplots(figsize=(7, 5.5))

# Color bars by rate
bar_colors = [OI[4] if r < 0.03 else OI[0] if r < 0.06 else OI[2] 
              for r in cmt_stats['enacted_rate']]

bars = ax.barh(range(len(cmt_stats)), cmt_stats['enacted_rate'] * 100,
               color=bar_colors, height=0.7, edgecolor='white', linewidth=0.3)

ax.set_yticks(range(len(cmt_stats)))
ax.set_yticklabels(cmt_stats['committee_nm'], fontsize=7.5)
ax.set_xlabel('Enacted rate (%)')

# Annotate with rate and bill count
for i, (_, row) in enumerate(cmt_stats.iterrows()):
    ax.text(row['enacted_rate'] * 100 + 0.3, i,
            f"{row['enacted_rate']*100:.1f}%  (n={row['n_bills']:,})",
            va='center', fontsize=6.5)

# Overall average line
avg = recent['enacted'].mean() * 100
ax.axvline(avg, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.text(avg + 0.2, len(cmt_stats) - 0.5, f'Avg: {avg:.1f}%', fontsize=7, color='gray')

ax.set_xlim(0, cmt_stats['enacted_rate'].max() * 100 * 1.6)

plt.tight_layout()
plt.show()

---
## 6. Legislative Inflation

As the number of bills introduced has soared, many are near-duplicates: identical bill names resubmitted by different legislators. This section examines the divergence between total bills and unique bill titles.

In [ ]:
# Total vs unique bill names per assembly (법률안 only)
inflation_data = []
for age in range(17, 23):
    sub = all_bills[(all_bills['age'] == age) & (all_bills['bill_kind'] == '법률안')]
    inflation_data.append({
        'age': age,
        'total': len(sub),
        'unique_names': sub['bill_nm'].nunique(),
    })

infl = pd.DataFrame(inflation_data)
infl['unique_pct'] = (infl['unique_names'] / infl['total'] * 100).round(1)

fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax2 = ax1.twinx()

x = np.arange(len(infl))
w = 0.35

ax1.bar(x - w/2, infl['total'], w, color=OI[3], alpha=0.85,
        label='Total bills', edgecolor='white', linewidth=0.5)
ax1.bar(x + w/2, infl['unique_names'], w, color=OI[0], alpha=0.85,
        label='Unique bill names', edgecolor='white', linewidth=0.5)

# Overlay line for unique percentage
line = ax2.plot(x, infl['unique_pct'], 'D-', color=OI[4],
               markersize=5, linewidth=1.5, label='Unique name share (%)')

ax1.set_xticks(x)
ax1.set_xticklabels([f'{a}대' for a in infl['age']])
ax1.set_ylabel('Count (법률안)')
ax2.set_ylabel('Unique name share (%)', color=OI[4])
ax2.tick_params(axis='y', labelcolor=OI[4])
ax2.set_ylim(0, 50)

# Combined legend
bars_handles = ax1.get_legend_handles_labels()
line_handles = ax2.get_legend_handles_labels()
ax1.legend(bars_handles[0] + line_handles[0],
           bars_handles[1] + line_handles[1],
           fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Top duplicated bill names (21대, the most recent complete assembly)
laws_21 = all_bills[(all_bills['age'] == 21) & (all_bills['bill_kind'] == '법률안')]
name_counts = laws_21['bill_nm'].value_counts()

top_dupes = name_counts.head(15).reset_index()
top_dupes.columns = ['Bill Name (법안명)', 'Count']
top_dupes.index = range(1, len(top_dupes) + 1)
top_dupes.index.name = 'Rank'
print(f'21대: {len(laws_21):,} 법률안, {laws_21["bill_nm"].nunique():,} unique names')
print(f'Top 15 most duplicated bill names:')
top_dupes

While total bills increased 3.5x from the 17th to the 21st assembly (7,490 to 25,862), unique bill names increased only 1.2x (2,921 to 3,561). The share of unique names dropped from 39% to 14%, indicating that much of the legislative inflation comes from multiple legislators proposing identically named bills on the same topic.

---
## 7. Temporal Patterns

Bill submissions follow clear temporal patterns driven by the legislative calendar, electoral cycles, and session deadlines.

In [ ]:
# Monthly bill submissions by assembly (line chart)
# Use month-of-term (0 = first month) to align assemblies
all_bills_ts = all_bills.dropna(subset=['ppsl_dt']).copy()

# Assembly start dates (approximate opening month)
assembly_starts = {
    17: pd.Timestamp('2004-05-01'),
    18: pd.Timestamp('2008-05-01'),
    19: pd.Timestamp('2012-05-01'),
    20: pd.Timestamp('2016-05-01'),
    21: pd.Timestamp('2020-05-01'),
    22: pd.Timestamp('2024-05-01'),
}

fig, ax = plt.subplots(figsize=(8, 4))
age_colors = {17: OI[5], 18: OI[0], 19: OI[2], 20: OI[1], 21: OI[3], 22: OI[4]}

for age in range(17, 23):
    sub = all_bills_ts[all_bills_ts['age'] == age].copy()
    monthly = sub.set_index('ppsl_dt').resample('ME').size()
    # Compute months since assembly start
    start = assembly_starts[age]
    months_since = ((monthly.index.year - start.year) * 12 +
                    (monthly.index.month - start.month))
    ax.plot(months_since, monthly.values, '-', color=age_colors[age],
            alpha=0.7, linewidth=1.0, label=f'{age}대')

ax.set_xlabel('Months since assembly opening')
ax.set_ylabel('Bills submitted per month')
ax.set_xlim(0, 48)
ax.legend(fontsize=7, ncol=3)

# Mark typical session-end months
for month_idx in [7, 19, 31, 43]:  # approximate December positions
    ax.axvline(month_idx, color='gray', linestyle=':', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Month-of-year distribution (all assemblies combined)
all_bills_ts['month'] = all_bills_ts['ppsl_dt'].dt.month
all_bills_ts['dow'] = all_bills_ts['ppsl_dt'].dt.dayofweek  # 0=Mon

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

# (a) Month distribution
month_counts = all_bills_ts.groupby(['age', 'month']).size().unstack(level=0, fill_value=0)
month_avg = month_counts.mean(axis=1)

colors_month = [OI[4] if m in [11, 12] else OI[3] for m in range(1, 13)]
axes[0].bar(range(1, 13), month_avg, color=colors_month,
            edgecolor='white', linewidth=0.3)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=7)
axes[0].set_ylabel('Average bills per month')
axes[0].annotate('Deadline\neffect', xy=(11.5, month_avg.loc[11]),
                 xytext=(9, month_avg.loc[11] * 1.1),
                 fontsize=7, color=OI[4],
                 arrowprops=dict(arrowstyle='->', color=OI[4], lw=0.8))

# (b) Day of week distribution
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_counts = all_bills_ts['dow'].value_counts().sort_index()

colors_dow = [OI[3]] * 5 + [OI[5]] * 2
axes[1].bar(range(7), dow_counts.values, color=colors_dow,
            edgecolor='white', linewidth=0.3)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(dow_labels, fontsize=7)
axes[1].set_ylabel('Total bills')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for ax, label in zip(axes, ['(a) Month of year', '(b) Day of week']):
    ax.set_title(label, fontsize=9, loc='left', fontweight='bold')

plt.tight_layout()
plt.show()

November and December show a pronounced deadline effect as legislators rush to file bills before the regular session ends. The day-of-week pattern shows that bill submissions are concentrated on weekdays, with near-zero activity on weekends.

---
## 8. DW-NOMINATE Ideal Points

We estimate DW-NOMINATE ideal points from member-level roll call votes for assemblies 20-22. The `coord1D` dimension captures the primary axis of legislative conflict (progressive-conservative in the Korean context). The `aligned` variable adjusts the sign so that positive values consistently indicate conservative alignment across terms.

In [ ]:
# Load DW-NOMINATE ideal points
dw = pd.read_csv(f'{DATA_DIR}/dw_ideal_points_20_22.csv')
print(f'DW-NOMINATE: {len(dw):,} legislator-terms')
print(f'Terms: {dw["term"].value_counts().sort_index().to_dict()}')
print(f'Party blocs: {dw["party_bloc"].value_counts().to_dict()}')
print(f'\ncoord1D range: [{dw["coord1D"].min():.3f}, {dw["coord1D"].max():.3f}]')
print(f'aligned range: [{dw["aligned"].min():.3f}, {dw["aligned"].max():.3f}]')

In [ ]:
# Voteview-style scatter: ideal points by assembly
bloc_colors = {
    'liberal': '#1f77b4',
    'conservative': '#d62728',
    'independent': '#7f7f7f',
    'left': '#2ca02c',
    'new_liberal': '#17becf',
    'centrist': '#ff7f0e',
    'other': '#bcbd22',
}

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), sharey=True)

for i, term in enumerate([20, 21, 22]):
    ax = axes[i]
    sub = dw[dw['term'] == term].copy()
    
    # Jitter y-axis for visibility
    np.random.seed(42)
    sub = sub.copy()
    sub['jitter'] = np.random.uniform(-0.3, 0.3, len(sub))
    
    for bloc, color in bloc_colors.items():
        mask = sub['party_bloc'] == bloc
        if mask.sum() > 0:
            ax.scatter(sub.loc[mask, 'aligned'], sub.loc[mask, 'jitter'],
                       s=12, alpha=0.6, color=color, label=bloc, edgecolors='none')
    
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_xlabel('DW-NOMINATE (aligned)', fontsize=8)
    ax.set_title(f'{term}대 (n={len(sub)})', fontsize=9, fontweight='bold')
    ax.set_xlim(-1.1, 1.1)
    ax.set_yticks([])

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, fontsize=7, ncol=4, loc='lower center',
           bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.show()

In [ ]:
# Party bloc mean ideal points by assembly
bloc_means = (dw.groupby(['term', 'party_bloc'])['aligned']
              .agg(['mean', 'std', 'count'])
              .reset_index())
bloc_means = bloc_means[bloc_means['count'] >= 5]  # Minimum group size

fig, ax = plt.subplots(figsize=(6, 3.5))
bloc_order = ['liberal', 'left', 'new_liberal', 'centrist', 'independent', 'conservative']
x_positions = {bloc: i for i, bloc in enumerate(bloc_order)}

for term, marker in zip([20, 21, 22], ['o', 's', 'D']):
    sub = bloc_means[bloc_means['term'] == term]
    for _, row in sub.iterrows():
        if row['party_bloc'] in x_positions:
            xp = x_positions[row['party_bloc']]
            offset = (term - 21) * 0.15  # Slight offset per term
            ax.errorbar(xp + offset, row['mean'], yerr=row['std'],
                        fmt=marker, markersize=6, capsize=3,
                        color=bloc_colors.get(row['party_bloc'], 'gray'),
                        alpha=0.8, label=f'{term}대' if row['party_bloc'] == 'liberal' else '')

ax.set_xticks(range(len(bloc_order)))
ax.set_xticklabels(bloc_order, fontsize=7.5, rotation=20, ha='right')
ax.set_ylabel('Mean ideal point (aligned)')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Polarization: gap between liberal and conservative means
polarization = []
for term in [20, 21, 22]:
    sub = dw[dw['term'] == term]
    lib = sub[sub['party_bloc'] == 'liberal']['aligned'].mean()
    con = sub[sub['party_bloc'] == 'conservative']['aligned'].mean()
    
    # Also compute with raw coord1D (W-NOMINATE equivalent)
    lib_w = sub[sub['party_bloc'] == 'liberal']['coord1D'].mean()
    con_w = sub[sub['party_bloc'] == 'conservative']['coord1D'].mean()
    
    polarization.append({
        'term': term,
        'liberal_mean': lib,
        'conservative_mean': con,
        'dw_gap': abs(con - lib),
        'w_gap': abs(con_w - lib_w),
    })

pol_df = pd.DataFrame(polarization)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar([f'{t}대' for t in pol_df['term']], pol_df['dw_gap'],
       width=0.4, color=OI[4], alpha=0.85, label='DW (aligned)', edgecolor='white')
ax.bar([f'{t}대' for t in pol_df['term']], pol_df['w_gap'],
       width=0.4, color=OI[1], alpha=0.4, label='W (coord1D)', edgecolor='white')

ax.set_ylabel('Polarization gap\n(|conservative mean - liberal mean|)')
ax.legend(fontsize=7)

for i, row in pol_df.iterrows():
    ax.text(i, row['dw_gap'] + 0.02, f"{row['dw_gap']:.2f}",
            ha='center', fontsize=8, color=OI[4])

plt.tight_layout()
plt.show()

---
## 9. Roll Call Votes

The roll call dataset contains 2.4M member-level vote records across assemblies 16-22. Votes come from three sources: the official API (`api`), PDF appendix extraction (`pdf_appendix`), and inline text parsing (`inline_text`). The bulk of coverage is in assemblies 20-22.

In [ ]:
# Load roll calls
rc = pd.read_parquet(f'{DATA_DIR}/roll_calls_all.parquet')
print(f'Roll call votes: {len(rc):,} rows')
print(f'Terms covered: {sorted(rc["term"].dropna().unique())}')
print(f'\nVote distribution:')
print(rc['vote'].value_counts().to_string())
print(f'\nSource distribution:')
print(rc['source'].value_counts().to_string())

In [ ]:
# Votes by assembly and source
rc_by_term = rc.groupby(['term', 'source']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# (a) Stacked bar: votes by term and source
source_colors = {'api': OI[3], 'pdf_appendix': OI[0], 'inline_text': OI[5]}
bottom = np.zeros(len(rc_by_term))
labels = [f'{int(t)}대' for t in rc_by_term.index]

for source in ['api', 'pdf_appendix', 'inline_text']:
    if source in rc_by_term.columns:
        vals = rc_by_term[source].values
        axes[0].bar(labels, vals, bottom=bottom, label=source,
                    color=source_colors[source], edgecolor='white', linewidth=0.3)
        bottom += vals

axes[0].set_ylabel('Number of vote records')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x/1000)}K'))
axes[0].legend(fontsize=7)
axes[0].tick_params(axis='x', labelsize=7.5)

# (b) Vote distribution (20-22대 only, where most data is)
rc_recent = rc[rc['term'].isin([20, 21, 22])]
vote_by_term = rc_recent.groupby(['term', 'vote']).size().unstack(fill_value=0)
vote_pct = vote_by_term.div(vote_by_term.sum(axis=1), axis=0) * 100

vote_colors = {'찬성': OI[2], '반대': OI[4], '기권': OI[0], '불참': OI[5]}
bottom = np.zeros(len(vote_pct))
labels_vote = [f'{int(t)}대' for t in vote_pct.index]

for vote_type in ['찬성', '반대', '기권', '불참']:
    if vote_type in vote_pct.columns:
        vals = vote_pct[vote_type].values
        axes[1].bar(labels_vote, vals, bottom=bottom, label=vote_type,
                    color=vote_colors[vote_type], edgecolor='white', linewidth=0.3)
        bottom += vals

axes[1].set_ylabel('Share (%)')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=7, loc='upper right')

for ax, label in zip(axes, ['(a) Vote records by assembly & source',
                             '(b) Vote type distribution (20-22대)']):
    ax.set_title(label, fontsize=9, loc='left', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Number of unique vote events and unique members
for term in sorted(rc['term'].dropna().unique()):
    sub = rc[rc['term'] == term]
    n_events = sub['vote_event'].nunique()
    n_members = sub['member_id'].nunique()
    n_records = len(sub)
    print(f'{int(term)}대: {n_records:>10,} vote records | '
          f'{n_events:>5,} vote events | {n_members:>4,} unique members')

---
## 10. Combining Ideal Points with Bill Lifecycle

The `rst_mona_cd` field in the bill master identifies the lead proposer (대표발의자) using a stable member identifier. This key links to `member_id` in the DW-NOMINATE dataset, enabling analysis of how a legislator's ideological position relates to their legislative success.

In [ ]:
# Merge DW-NOMINATE with bill data
# Load full masters for 20-22 (they have rst_mona_cd)
bills_recent = []
for age in [20, 21, 22]:
    df = pd.read_parquet(f'{DATA_DIR}/master_bills_{age}.parquet')
    bills_recent.append(df)

bills_rc = pd.concat(bills_recent, ignore_index=True)

# Keep only legislator-sponsored bills (의원) with a lead proposer code
bills_rc = bills_rc[
    (bills_rc['ppsr_kind'] == '의원') & (bills_rc['rst_mona_cd'].notna())
].copy()

# Merge with DW-NOMINATE via rst_mona_cd = member_id
# Match on both member_id and term
bills_rc = bills_rc.rename(columns={'rst_mona_cd': 'member_id'})
bills_rc['term'] = bills_rc['age']  # age = term for merging

merged = bills_rc.merge(
    dw[['member_id', 'term', 'aligned', 'party_bloc', 'coord1D']],
    on=['member_id', 'term'],
    how='inner'
)

print(f'Bills with DW-NOMINATE scores: {len(merged):,}')
print(f'Unique legislators: {merged["member_id"].nunique()}')
print(f'Assembly distribution:')
print(merged.groupby('age').size().to_string())

In [ ]:
# Ideological extremism vs. passage rate
merged['extremism'] = merged['aligned'].abs()

# Bin extremism into deciles
merged['ext_bin'] = pd.qcut(merged['extremism'], 10, labels=False, duplicates='drop')

ext_rates = merged.groupby('ext_bin').agg(
    mean_extremism=('extremism', 'mean'),
    enacted_rate=('enacted', 'mean'),
    n_bills=('bill_id', 'size'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

# (a) Extremism vs enacted rate
axes[0].scatter(ext_rates['mean_extremism'], ext_rates['enacted_rate'] * 100,
                s=ext_rates['n_bills'] / 30, color=OI[3], alpha=0.7, edgecolors='white')

# Fit a simple trend line
z = np.polyfit(ext_rates['mean_extremism'], ext_rates['enacted_rate'] * 100, 1)
p = np.poly1d(z)
x_line = np.linspace(ext_rates['mean_extremism'].min(), ext_rates['mean_extremism'].max(), 50)
axes[0].plot(x_line, p(x_line), '--', color=OI[4], linewidth=1.2, alpha=0.8)

axes[0].set_xlabel('Ideological extremism (|aligned|)')
axes[0].set_ylabel('Enacted rate (%)')
axes[0].text(0.95, 0.95, f'Bubble size = n bills', transform=axes[0].transAxes,
             ha='right', va='top', fontsize=7, color='gray')

# (b) Extremism vs processing time (for processed bills)
processed = merged[merged['days_to_proc'].notna() & (merged['days_to_proc'] > 0)].copy()
proc_ext = processed.groupby('ext_bin').agg(
    mean_extremism=('extremism', 'mean'),
    median_days=('days_to_proc', 'median'),
    n_bills=('bill_id', 'size'),
).reset_index()

axes[1].scatter(proc_ext['mean_extremism'], proc_ext['median_days'],
                s=proc_ext['n_bills'] / 10, color=OI[0], alpha=0.7, edgecolors='white')

# Trend line
z2 = np.polyfit(proc_ext['mean_extremism'], proc_ext['median_days'], 1)
p2 = np.poly1d(z2)
axes[1].plot(x_line, p2(x_line), '--', color=OI[4], linewidth=1.2, alpha=0.8)

axes[1].set_xlabel('Ideological extremism (|aligned|)')
axes[1].set_ylabel('Median days to decision')

for ax, label in zip(axes, ['(a) Extremism vs. passage rate',
                             '(b) Extremism vs. processing time']):
    ax.set_title(label, fontsize=9, loc='left', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Party bloc comparison: enacted rates with ideal point context
bloc_legislative = merged.groupby('party_bloc').agg(
    n_bills=('bill_id', 'size'),
    enacted_rate=('enacted', 'mean'),
    mean_extremism=('extremism', 'mean'),
    mean_aligned=('aligned', 'mean'),
).reset_index()
bloc_legislative = bloc_legislative[bloc_legislative['n_bills'] >= 50]
bloc_legislative['enacted_rate'] = (bloc_legislative['enacted_rate'] * 100).round(1)
bloc_legislative['mean_extremism'] = bloc_legislative['mean_extremism'].round(3)
bloc_legislative['mean_aligned'] = bloc_legislative['mean_aligned'].round(3)
bloc_legislative = bloc_legislative.sort_values('enacted_rate', ascending=False)
bloc_legislative.columns = ['Party Bloc', 'N Bills', 'Enacted Rate (%)',
                             'Mean Extremism', 'Mean Ideal Point']
bloc_legislative

---
## Notes

- **22대 is in session**: The 22nd National Assembly opened on 2024-05-30 and is ongoing. Cross-assembly comparisons should account for the fact that many bills are still pending.
- **`passed` vs `enacted`**: `passed` counts 대안반영폐기 (content absorbed into a committee omnibus alternative) as passage. `enacted` only counts bills that themselves became law (원안가결 + 수정가결).
- **Committee meetings**: `committee_meetings_{age}` and `judiciary_meetings_{age}` provide bill-meeting-level procedural records for each assembly.
- **Roll calls**: The `roll_calls_all.parquet` file covers assemblies 16-22, with comprehensive coverage from the 20th assembly onward.
- **DW-NOMINATE**: Ideal points are estimated from roll call votes for assemblies 20-22 (936 legislator-terms). The `aligned` variable adjusts sign direction for cross-term comparability.
- **Join keys**: `bill_id` is the primary key across all bill tables. `rst_mona_cd` (lead proposer code) and `member_id` link to legislator-level datasets.

**Data source**: 열린국회정보 Open API (https://open.assembly.go.kr)